# 疑似データ（PerSearch の camera 再現）の準備

疑似データ（PerSearch の camera 再現ベンチマーク結果）を、camera 実データと直接比較できる
「生の0/1・ドル価格」符号化の long CSV に変換する。

`data/persearch_camera.xlsx`（PerSearch で `camera` と同じ7属性・価格5水準
（$79/$129/$179/$229/$279）の調査を実施した結果）の「回答データ（Long形式）」
シートを読み、`bayesm::camera` の生の X 符号化（canon/sony/nikon/panasonic の
4ダミー・pixels/zoom/video/swivel/wifi の2値ダミー・price を100ドル単位の
連続値）に揃えた long CSV を出力する。

参照: `prepare_camera.ipynb`（camera 側の同形式変換）、`hb_diag.ipynb`（本notebookの
出力を読んで対角共分散の階層ベイズMNLを推定する）。


In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd


In [ ]:
DATA_DIR = Path.cwd().parent / "data"
RESULTS_DIR = Path.cwd().parent / "results"

XLSX_PATH = DATA_DIR / "persearch_camera.xlsx"
OUTPUT_PATH = RESULTS_DIR / "persearch_camera_raw_long.csv"
LABELED_OUTPUT_PATH = RESULTS_DIR / "persearch_camera_long.csv"
SHEET_NAME = "回答データ（Long形式）"

# PerSearch 側の水準文字列 -> camera 側の raw 符号化（0=基準水準/1=上位水準）
BRAND_MAP = {"キヤノン": "canon", "ソニー": "sony", "ニコン": "nikon", "パナソニック": "panasonic"}
BINARY_MAPS = {
    "画素数": {"標準画素数": 0, "高画素数": 1},
    "ズーム": {"標準ズーム": 0, "高倍率ズーム": 1},
    "動画撮影": {"動画非対応": 0, "動画対応": 1},
    "可動式モニタ": {"可動式モニタなし": 0, "可動式モニタあり": 1},
    "Wi-Fi": {"Wi-Fi非対応": 0, "Wi-Fi対応": 1},
}
BINARY_RAW_COLS = {
    "画素数": "pixels",
    "ズーム": "zoom",
    "動画撮影": "video",
    "可動式モニタ": "swivel",
    "Wi-Fi": "wifi",
}
# PerSearch 側の水準文字列 -> camera_long.csv と同じカテゴリ文字列（counting.ipynb がそのまま使える表記）
LABEL_MAPS = {
    "画素数": {"標準画素数": "低画素", "高画素数": "高画素"},
    "ズーム": {"標準ズーム": "標準ズーム", "高倍率ズーム": "高倍率ズーム"},
    "動画撮影": {"動画非対応": "動画機能なし", "動画対応": "動画機能あり"},
    "可動式モニタ": {"可動式モニタなし": "回転式液晶なし", "可動式モニタあり": "回転式液晶あり"},
    "Wi-Fi": {"Wi-Fi非対応": "Wi-Fiなし", "Wi-Fi対応": "Wi-Fiあり"},
}
LABEL_RAW_COLS = BINARY_RAW_COLS
PRICE_COLUMN_JA = "価格"
PRICE_RE = re.compile(r"\d+")

RAW_COLUMNS = [
    "canon", "sony", "nikon", "panasonic",
    "pixels", "zoom", "video", "swivel", "wifi",
    "price_usd100",
]


## 関数定義

In [ ]:
def parse_price_usd100(raw: str | float) -> float:
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return 0.0
    m = PRICE_RE.search(str(raw))
    return float(m.group(0)) / 100.0 if m else 0.0


In [ ]:
def load_long_sheet() -> pd.DataFrame:
    return pd.read_excel(XLSX_PATH, sheet_name=SHEET_NAME)


In [ ]:
def build_raw_long_df(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.copy()

    # uuid（ペルソナID）を出現順の連番 respondent_id に変換する
    uuid_order = df["uuid"].drop_duplicates().tolist()
    uuid_to_id = {u: i + 1 for i, u in enumerate(uuid_order)}
    df["respondent_id"] = df["uuid"].map(uuid_to_id)

    is_product = df["is_none"] == 0

    # ブランド: 4列のダミー（none行はすべて0）
    for ja, en in BRAND_MAP.items():
        df[en] = ((df["ブランド"] == ja) & is_product).astype(float)

    # 2値属性: 0/1（none行はすべて0）
    for ja_col, raw_col in BINARY_RAW_COLS.items():
        mapped = df[ja_col].map(BINARY_MAPS[ja_col])
        df[raw_col] = mapped.where(is_product, 0.0).fillna(0.0).astype(float)

    # 価格: 100ドル単位の連続値（none行は0）
    df["price_usd100"] = df[PRICE_COLUMN_JA].apply(parse_price_usd100)
    df.loc[~is_product, "price_usd100"] = 0.0

    keep_cols = ["respondent_id", "task_index", "option_label", "is_none", "chosen"] + RAW_COLUMNS
    out = df[keep_cols].sort_values(["respondent_id", "task_index"]).reset_index(drop=True)
    return out


In [ ]:
def build_labeled_long_df(raw: pd.DataFrame) -> pd.DataFrame:
    """camera_long.csv と同じカテゴリ文字列表記の long CSV を作る（counting.ipynb 用）。"""
    df = raw.copy()

    uuid_order = df["uuid"].drop_duplicates().tolist()
    uuid_to_id = {u: i + 1 for i, u in enumerate(uuid_order)}
    df["respondent_id"] = df["uuid"].map(uuid_to_id)

    is_product = df["is_none"] == 0

    df["brand"] = df["ブランド"].map(BRAND_MAP)
    df.loc[~is_product, "brand"] = None

    for ja_col, out_col in LABEL_RAW_COLS.items():
        df[out_col] = df[ja_col].map(LABEL_MAPS[ja_col])
        df.loc[~is_product, out_col] = None

    df["price_usd100"] = df[PRICE_COLUMN_JA].apply(parse_price_usd100)
    df.loc[~is_product, "price_usd100"] = None

    keep_cols = [
        "respondent_id", "task_index", "option_label", "is_none", "chosen",
        "brand", "pixels", "zoom", "video", "swivel", "wifi", "price_usd100",
    ]
    out = df[keep_cols].sort_values(["respondent_id", "task_index"]).reset_index(drop=True)
    return out


In [ ]:
def validate(df: pd.DataFrame) -> None:
    n_resp = df["respondent_id"].nunique()
    n_tasks = df["task_index"].nunique()
    n_alt = len(df) // (n_resp * n_tasks)
    assert len(df) == n_resp * n_tasks * n_alt, "row count mismatch"

    chosen_per_task = df.groupby(["respondent_id", "task_index"])["chosen"].sum()
    assert (chosen_per_task == 1).all(), "each task must have exactly one chosen row"

    none_per_task = df.groupby(["respondent_id", "task_index"])["is_none"].sum()
    assert (none_per_task == 1).all(), "each task must have exactly one none row"

    # 積(none行を除く)がブランドはちょうど1つ立っている
    brand_sum = df.loc[df["is_none"] == 0, ["canon", "sony", "nikon", "panasonic"]].sum(axis=1)
    assert (brand_sum == 1).all(), "each product row must have exactly one brand flag set"


## 実行

`data/persearch_camera.xlsx` が存在すること。

In [ ]:
if not XLSX_PATH.exists():
    raise SystemExit(f"{XLSX_PATH} が見つかりません。")

raw = load_long_sheet()
df = build_raw_long_df(raw)
validate(df)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

n_resp = df["respondent_id"].nunique()
n_tasks = df["task_index"].nunique()
n_alt = len(df) // (n_resp * n_tasks)
print(f"wrote {len(df)} rows to {OUTPUT_PATH}")
print(f"respondents: {n_resp}, tasks/respondent: {n_tasks}, alts/task: {n_alt}")
print("none choice rate:", df.loc[df["is_none"] == 1, "chosen"].mean())
print("price levels (USD/100):", sorted(df["price_usd100"].unique()))

labeled = build_labeled_long_df(raw)
labeled.to_csv(LABELED_OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"wrote {len(labeled)} rows to {LABELED_OUTPUT_PATH}")
